<a href="https://colab.research.google.com/github/SaanchitaV/RagAllied-SARAGA-Dataset/blob/main/RagaAllied_Phrase_Probing_Full_SARAGA_Carnatic_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install mirdata compiam pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 47.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 10.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of tifffile to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.9/275.9 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.6/260.6 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.

In [1]:
import mirdata
ds = mirdata.initialize("saraga_carnatic", data_home="/content/saraga_carnatic", version="1.5")
print(ds.remotes.keys())

dict_keys(['all'])


In [2]:
"""
Saraga Carnatic v1.5 -- download + mirdata census + phrase-coverage probe
=============================================================
Full-dataset (249-track) pre-flight, all in one runnable cell.

Handles: downloading via mirdata (~14GB, once), checksum validation, coverage
of tonic/pitch/pitch_vocal/phrases, FRAME RATE for Section 3, phrase solfege
vocabulary (phrase-representation viability), raga/artist distribution.

STORAGE (Colab): the 14GB audio is INPUT; the extracted sequences are the
artifact. Download to ephemeral /content (fine for one session), then save only
the small outputs to Drive. With paid Drive (>30GB), set DATA_HOME to a Drive
path to persist the audio.

Set DOWNLOAD=False on later runs in the same session.
"""

import os
import csv
from collections import Counter

import numpy as np
import mirdata

# ============ CONFIG ============
DATA_HOME = "/content/saraga_carnatic"   # ephemeral; Drive path to persist
DOWNLOAD = True                          # False if already downloaded this session
OUT_DIR = "/content/census_out"
os.makedirs(OUT_DIR, exist_ok=True)

ds = mirdata.initialize("saraga_carnatic", data_home=DATA_HOME, version="1.5")

# ---- download ----
if DOWNLOAD:
    print("Downloading Saraga Carnatic v1.5 (~14GB, one time)...")
    try:
        ds.download(cleanup=True)
    except Exception as e:
        print(f"download() raised: {e}")
        try:
            print("valid remote keys:", list(ds.remotes.keys()))
        except Exception:
            pass
        raise

# ---- validate ----
print("\nValidating (checksums)...")
missing, invalid = ds.validate(verbose=False)
print(f"  missing: {len(missing)}   invalid: {len(invalid)}")

track_ids = ds.track_ids
n = len(track_ids)
print(f"\nTotal tracks: {n}")

print("\n--- CITATION ---")
try: ds.cite()
except Exception: pass
print("\n--- LICENSE ---")
try: ds.license()
except Exception: pass

# ============ PROBE ============
has_tonic = has_pitch = has_pitch_vocal = has_phrases = has_sections = 0
frame_hops = []
phrase_symbols = Counter()
phrase_ngrams = Counter()
raga_counter = Counter()
artist_counter = Counter()
multi_raga = 0
rows = []

for tid in track_ids:
    t = ds.track(tid)
    tonic_ok = pitch_ok = pv_ok = ph_ok = sec_ok = False

    try: tonic_ok = t.tonic is not None
    except Exception: pass

    try:
        if t.pitch is not None and len(t.pitch.times) > 1:
            pitch_ok = True
            frame_hops.append(float(t.pitch.times[1] - t.pitch.times[0]))
    except Exception: pass

    try: pv_ok = t.pitch_vocal is not None and len(t.pitch_vocal.times) > 1
    except Exception: pass

    try:
        if t.phrases is not None and len(t.phrases.intervals) > 0:
            ph_ok = True
            for lab in t.phrases.labels:
                s = str(lab)
                phrase_ngrams[s] += 1
                for ch in s:
                    if ch in "SsRrGgMmPpDdNn":
                        phrase_symbols[ch] += 1
    except Exception: pass

    try: sec_ok = t.sections is not None and len(t.sections.intervals) > 0
    except Exception: pass

    has_tonic += tonic_ok; has_pitch += pitch_ok; has_pitch_vocal += pv_ok
    has_phrases += ph_ok; has_sections += sec_ok

    raga_names, artist_name = [], "UNKNOWN"
    try:
        md = t.metadata or {}
        raagas = md.get("raaga") or []
        raga_names = [r.get("name", "?") for r in raagas if isinstance(r, dict)]
        if len(raga_names) > 1: multi_raga += 1
        for r in raga_names: raga_counter[r] += 1
        arts = md.get("album_artists") or md.get("artists") or []
        if arts and isinstance(arts[0], dict):
            artist_name = arts[0].get("name", "UNKNOWN")
        artist_counter[artist_name] += 1
    except Exception: pass

    rows.append({"track_id": tid, "tonic": tonic_ok, "pitch": pitch_ok,
                 "pitch_vocal": pv_ok, "phrases": ph_ok,
                 "n_ragas": len(raga_names), "artist": artist_name})

print(f"\n=== COVERAGE (of {n} tracks) ===")
print(f"  tonic       : {has_tonic:3d} ({100*has_tonic/n:.0f}%)")
print(f"  pitch (mix) : {has_pitch:3d} ({100*has_pitch/n:.0f}%)")
print(f"  pitch_vocal : {has_pitch_vocal:3d} ({100*has_pitch_vocal/n:.0f}%)  <- sung line")
print(f"  phrases     : {has_phrases:3d} ({100*has_phrases/n:.0f}%)  <- phrase-rep viability")
print(f"  sections    : {has_sections:3d} ({100*has_sections/n:.0f}%)")

if frame_hops:
    hop = float(np.median(frame_hops))
    print(f"\n=== FRAME RATE (Section 3) ===")
    print(f"  median pitch hop = {hop*1000:.2f} ms")
    print(f"  -> 21-frame median filter ~= {21*hop*1000:.0f} ms")
    print(f"  -> 10-frame min dwell     ~= {10*hop*1000:.0f} ms")

print(f"\n=== RAGA / ARTIST distribution ===")
print(f"  distinct ragas   : {len(raga_counter)}")
print(f"  distinct artists : {len(artist_counter)}")
print(f"  multi-raga (ragamalika): {multi_raga}")
print(f"  ragas >=2 recordings (evaluable): {sum(1 for c in raga_counter.values() if c >= 2)}")
print(f"  ragas >=3 recordings: {sum(1 for c in raga_counter.values() if c >= 3)}")

if phrase_symbols:
    print(f"\n=== PHRASE solfege vocabulary ===")
    for sym in "SsRrGgMmPpDdNn":
        if phrase_symbols[sym]:
            print(f"  {sym}: {phrase_symbols[sym]}")
    print(f"  distinct phrase strings: {len(phrase_ngrams)}")
    print("  (case = functional swara identity, NOT pitch-folded)")

with open(os.path.join(OUT_DIR, "saraga_v15_census.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)
print(f"\nSaved per-track census -> {OUT_DIR}/saraga_v15_census.csv")

print("\nDECISIONS THIS INFORMS:")
print("  1. pitch vs pitch_vocal -> pick fully-covered one (prefer vocal)")
print("  2. phrase coverage high -> build phrase-level representation")
print("  3. these counts replace 197/176/95 in the paper")

13.4GB [31:37, 7.58MB/s]                            
544kB [00:01, 508kB/s]                            



Validating (checksums)...
  missing: 1   invalid: 1

Total tracks: 249

--- CITATION ---
========== BibTeX ==========

@dataset{bozkurt_b_2018_4301737,
  author       = {Bozkurt, B. and
                  Srinivasamurthy, A. and
                  Gulati, S. and
                  Serra, X.},
  title        = {Saraga: research datasets of Indian Art Music},
  month        = may,
  year         = 2018,
  publisher    = {Zenodo},
  version      = {1.5},
  doi          = {10.5281/zenodo.4301737},
  url          = {https://doi.org/10.5281/zenodo.4301737}
}


--- LICENSE ---
========== License ==========
Creative Commons Attribution Non Commercial Share Alike 4.0 International.

******************************************************************************************
DISCLAIMER: mirdata is a software package with its own license which is independent from
this dataset's license. We don not take responsibility for possible inaccuracies in the
license information provided in mirdata. It is the 

In [3]:
import os; print(os.path.exists("/content/saraga_carnatic"))

True


In [5]:
t0 = ds.track("14_Angakaram")
print([a for a in dir(t0.phrases) if not a.startswith("_")])

['event_unit', 'events', 'interval_unit', 'intervals']


In [6]:
"""
Phrase sub-study, Step 1 -- PROBE (patched for EventData: .events not .labels)
=============================================================
Reads the phrase annotations (EventData: .intervals for timing, .events for
solfege labels). Answers: raga distribution, raw label format, vocabulary size,
phrase length, per-track counts, and the KEY question -- do phrases recur
within a raga (same-raga Jaccard) more than across ragas?
"""

import os
import random
from collections import Counter, defaultdict

import numpy as np
import mirdata

DATA_HOME = "/content/saraga_carnatic"
ds = mirdata.initialize("saraga_carnatic", data_home=DATA_HOME, version="1.5")

def phrase_labels(t):
    """Return list of phrase label strings from an EventData object."""
    ph = t.phrases
    if ph is None:
        return []
    ev = getattr(ph, "events", None)
    if ev is None:
        return []
    return [str(e) for e in ev]

# ---- gather phrase-bearing tracks ----
phrase_tracks = []
for tid in ds.track_ids:
    t = ds.track(tid)
    try:
        labs = phrase_labels(t)
        if labs:
            raga = "?"
            md = t.metadata or {}
            raagas = md.get("raaga") or []
            if raagas and isinstance(raagas[0], dict):
                raga = raagas[0].get("name", "?")
            phrase_tracks.append((tid, labs, raga))
    except Exception:
        pass

print(f"Tracks with phrase annotations: {len(phrase_tracks)}")

# ---- (1) raga distribution (exclude ragamalika / unknown for evaluable) ----
raga_counter = Counter(r for _, _, r in phrase_tracks)
def evaluable_raga(r):
    return r not in ("Rāgamālika", "Ragamalika", "?", "")
ev_ragas = {r: c for r, c in raga_counter.items() if evaluable_raga(r)}
print(f"Distinct ragas: {len(raga_counter)}")
print(f"Evaluable ragas (>=2 tracks, excl ragamalika/unknown): "
      f"{sum(1 for r,c in ev_ragas.items() if c>=2)}")

# ---- (2) raw label examples ----
print("\n--- RAW phrase labels (first track) ---")
tid0, labs0, raga0 = phrase_tracks[0]
print(f"track: {tid0}  raga: {raga0}  ({len(labs0)} phrases)")
for lab in labs0[:15]:
    print(f"   {lab!r}")

# ---- (3) vocabulary, (4) length, (5) per-track counts ----
all_labels = Counter()
lengths, per_track = [], []
for tid, labs, raga in phrase_tracks:
    per_track.append(len(labs))
    for l in labs:
        all_labels[l] += 1
        lengths.append(len(l.replace(" ", "")))

print(f"\n--- VOCABULARY ---")
print(f"  total phrase instances : {sum(all_labels.values())}")
print(f"  distinct phrase strings: {len(all_labels)}")
print(f"  appearing >=2x : {sum(1 for c in all_labels.values() if c>=2)}")
print(f"  appearing >=5x : {sum(1 for c in all_labels.values() if c>=5)}")
print("  most common:")
for lab, c in all_labels.most_common(15):
    print(f"     {c:4d}x  {lab!r}")

if lengths:
    L = np.array(lengths)
    print(f"\n--- PHRASE LENGTH (chars) --- min {L.min()} median {int(np.median(L))} "
          f"mean {L.mean():.1f} max {L.max()}")
if per_track:
    P = np.array(per_track)
    print(f"--- PHRASES PER TRACK --- min {P.min()} median {int(np.median(P))} "
          f"mean {P.mean():.1f} max {P.max()}")

# ---- (6) recurrence: same-raga vs diff-raga phrase-set Jaccard ----
print(f"\n--- PHRASE RECURRENCE (the key gate) ---")
raga_to = defaultdict(list)
for tid, labs, raga in phrase_tracks:
    if evaluable_raga(raga):
        raga_to[raga].append((tid, set(labs)))

within = []
for rg, lst in raga_to.items():
    for i in range(len(lst)):
        for j in range(i+1, len(lst)):
            a, b = lst[i][1], lst[j][1]
            u = len(a | b)
            if u: within.append(len(a & b)/u)

flat = [(rg, phr) for rg, lst in raga_to.items() for _, phr in lst]
random.seed(42)
across = []
for _ in range(1000):
    x, y = random.choice(flat), random.choice(flat)
    if x[0] != y[0]:
        u = len(x[1] | y[1])
        if u: across.append(len(x[1] & y[1])/u)

if within:
    print(f"  same-raga Jaccard: mean {np.mean(within):.3f}  (n={len(within)} pairs)")
if across:
    print(f"  diff-raga Jaccard: mean {np.mean(across):.3f}  (n={len(across)} pairs)")
if within and across:
    ratio = np.mean(within)/max(np.mean(across),1e-9)
    print(f"  ratio within/across: {ratio:.2f}")
    print("  >>1 : characteristic phrases recur within raga -> build the representation")
    print("  ~=1 : phrases too track-specific (or transcription variance) -> "
          "try approximate matching before concluding")

Tracks with phrase annotations: 100
Distinct ragas: 58
Evaluable ragas (>=2 tracks, excl ragamalika/unknown): 22

--- RAW phrase labels (first track) ---
track: 14_Angakaram  raga: Suraṭi  (11 phrases)
   'rmpnnndsndp'
   'mgpmmgrms'
   'ndnsndpm'
   'ndnsndpm'
   'rrrpmgrgrssrn'
   'rrrpmgrgrssrn'
   'ndnsndppm'
   'mmgmpmmgr'
   'mmgmpmgr'
   'nnndnsndp'
   'nnnsndp'

--- VOCABULARY ---
  total phrase instances : 1474
  distinct phrase strings: 567
  appearing >=2x : 310
  appearing >=5x : 88
  most common:
       18x  'gmnns'
       15x  'srgrsnsrgrs'
       14x  'gmpgmpnmps'
       12x  'pgmdd'
       12x  'dmgrs'
       11x  'smpddndndpmpm'
       11x  'ggnpgr'
       11x  'mgpds'
       11x  'mpmgrgss'
       11x  'gmdnrsndmpgsrg'
       11x  'mrggrrss'
       10x  'sndpd'
       10x  'mgsdndp'
       10x  'ssdpmgmrsn'
       10x  'nmgrs'

--- PHRASE LENGTH (chars) --- min 0 median 8 mean 8.2 max 28
--- PHRASES PER TRACK --- min 1 median 11 mean 14.7 max 57

--- PHRASE RECURRENCE

In [7]:
"""
Phrase sub-study, Step 2 -- PATH 1: exact-phrase TF-IDF VSM retrieval
=============================================================
Each recording -> a bag of its expert-annotated phrase strings. Build a TF-IDF
vector-space model over the phrase vocabulary (the Gulati-style phrase VSM),
retrieve by cosine, evaluate with the SAME protocol as the main paper.

Why this could beat swara-counting: unlike the 12-swara vocabulary (where every
symbol occurs in every recording and IDF is degenerate), phrases are RARE and
raga-specific (probe: same-raga vs diff-raga Jaccard = 57x). So IDF actually
weights here -- discriminative phrases get high weight.

Evaluated on the 100 phrase-annotated tracks (22 evaluable ragas). This is a
SEPARATE sub-study from the main 43-raga comparison; report in its own table.
"""

import os
import random
from collections import Counter, defaultdict

import numpy as np
import mirdata

DATA_HOME = "/content/saraga_carnatic"
OUT_DIR = "/content/census_out"
os.makedirs(OUT_DIR, exist_ok=True)
ds = mirdata.initialize("saraga_carnatic", data_home=DATA_HOME, version="1.5")

EXCLUDE = ("Rāgamālika", "Ragamalika", "?", "")


def phrase_labels(t):
    ph = t.phrases
    if ph is None:
        return []
    ev = getattr(ph, "events", None)
    return [str(e) for e in ev] if ev is not None else []


# ---- collect phrase-annotated, labelled tracks ----
recs = []  # (track_id, [phrases], raga)
for tid in ds.track_ids:
    t = ds.track(tid)
    labs = phrase_labels(t)
    if not labs:
        continue
    raga = "?"
    md = t.metadata or {}
    raagas = md.get("raaga") or []
    if raagas and isinstance(raagas[0], dict):
        raga = raagas[0].get("name", "?")
    if raga in EXCLUDE:
        continue
    recs.append((tid, labs, raga))

ragas = [r for _, _, r in recs]
raga_counts = Counter(ragas)
evaluable = {r for r, c in raga_counts.items() if c >= 2}
print(f"Phrase-annotated labelled tracks: {len(recs)}")
print(f"Evaluable ragas (>=2 tracks): {len(evaluable)}")
print(f"Evaluable queries: {sum(1 for r in ragas if r in evaluable)}")


# ---- build TF-IDF over phrase vocabulary ----
def build_tfidf(docs):
    """docs: list of lists of phrase-strings -> (matrix, vocab)."""
    df = Counter()
    for d in docs:
        for p in set(d):
            df[p] += 1
    vocab = {p: i for i, p in enumerate(sorted(df))}
    N = len(docs)
    idf = np.zeros(len(vocab))
    for p, i in vocab.items():
        idf[i] = np.log((N + 1) / (df[p] + 1)) + 1     # smoothed idf
    X = np.zeros((N, len(vocab)))
    for r, d in enumerate(docs):
        tf = Counter(d)
        for p, c in tf.items():
            X[r, vocab[p]] = c * idf[vocab[p]]
    # L2 normalize
    norm = np.linalg.norm(X, axis=1, keepdims=True); norm[norm == 0] = 1
    return X / norm, vocab


docs = [labs for _, labs, _ in recs]
X, vocab = build_tfidf(docs)
print(f"Phrase vocabulary size: {len(vocab)}")


# ---- retrieval eval (same protocol as main paper) ----
def evaluate(X, ragas, evaluable, name, ks=(1, 3, 5)):
    sim = X @ X.T
    counts = Counter(ragas)
    precisions = {k: [] for k in ks}
    aps, rrs = [], []
    n_eval = 0
    for i in range(len(ragas)):
        if ragas[i] not in evaluable:
            continue
        n_eval += 1
        order = np.argsort(-sim[i]); order = order[order != i]
        rel = np.array([ragas[j] == ragas[i] for j in order], dtype=int)
        for k in ks:
            precisions[k].append(rel[:k].sum() / k)
        if rel.sum():
            cum = np.cumsum(rel)
            prec_at = cum / (np.arange(len(rel)) + 1)
            aps.append((prec_at * rel).sum() / rel.sum())
            rrs.append(1.0 / (np.argmax(rel) + 1))
    row = {"representation": name, "evaluable_queries": n_eval}
    for k in ks:
        row[f"P@{k}"] = round(float(np.mean(precisions[k])), 3)
    row["MAP"] = round(float(np.mean(aps)), 3)
    row["MRR"] = round(float(np.mean(rrs)), 3)
    return row


# ---- also compute bag-of-notes ON THE SAME 100-track set for a fair comparison ----
# (phrases are strings of solfege chars; derive a swara-count vector from them
#  so both representations use the SAME underlying data)
SWARA_CHARS = "SsRrGgMmPpDdNn"
def bag_of_notes_from_phrases(labs):
    v = Counter()
    for p in labs:
        for ch in p:
            if ch in SWARA_CHARS:
                v[ch] += 1
    vec = np.array([v[c] for c in SWARA_CHARS], dtype=float)
    s = vec.sum()
    return vec / s if s else vec

Xbon = np.array([bag_of_notes_from_phrases(labs) for _, labs, _ in recs])
nb = np.linalg.norm(Xbon, axis=1, keepdims=True); nb[nb == 0] = 1
Xbon = Xbon / nb

import pandas as pd
results = [
    evaluate(Xbon, ragas, evaluable, "bag_of_notes (from phrase chars)"),
    evaluate(X,    ragas, evaluable, "phrase_tfidf_vsm (exact)"),
]
df = pd.DataFrame(results)
print("\n=== PHRASE SUB-STUDY: retrieval on 100 phrase-annotated tracks ===")
print(df.to_string(index=False))
print("\nKey comparison: does phrase_tfidf_vsm beat bag_of_notes ON THE SAME tracks?")
print("(This is the fair test -- both built from the same phrase-annotated data.)")
df.to_csv(os.path.join(OUT_DIR, "phrase_vsm_results.csv"), index=False)
print(f"Saved -> {OUT_DIR}/phrase_vsm_results.csv")

Phrase-annotated labelled tracks: 94
Evaluable ragas (>=2 tracks): 22
Evaluable queries: 60
Phrase vocabulary size: 535

=== PHRASE SUB-STUDY: retrieval on 100 phrase-annotated tracks ===
                  representation  evaluable_queries   P@1   P@3   P@5   MAP   MRR
bag_of_notes (from phrase chars)                 60 0.150 0.122 0.083 0.212 0.276
        phrase_tfidf_vsm (exact)                 60 0.333 0.139 0.100 0.291 0.391

Key comparison: does phrase_tfidf_vsm beat bag_of_notes ON THE SAME tracks?
(This is the fair test -- both built from the same phrase-annotated data.)
Saved -> /content/census_out/phrase_vsm_results.csv


In [8]:
"""
Phrase sub-study, Step 3 -- PATH 2: approximate phrase matching, then VSM
=============================================================
Exact-match VSM (path 1) treats 'ndnsndpm' and 'ndnsndppm' as different terms,
so transcription variance suppresses same-raga overlap. Path 2 first CLUSTERS
near-duplicate phrases into canonical terms (edit-distance / normalized
similarity -- the same fuzzy-matching idea used for BOM record linkage), then
rebuilds the TF-IDF VSM over clustered terms.

The delta (path2 - path1) quantifies how much transcription variance was
costing the phrase representation. If path2 >> path1: phrases are bottlenecked
by transcription consistency, not by being uninformative.

Compares three on the SAME 94-track / 60-query set:
  bag_of_notes (chars) | phrase_vsm_exact | phrase_vsm_approx
"""

import os
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import mirdata

DATA_HOME = "/content/saraga_carnatic"
OUT_DIR = "/content/census_out"
ds = mirdata.initialize("saraga_carnatic", data_home=DATA_HOME, version="1.5")

EXCLUDE = ("Rāgamālika", "Ragamalika", "?", "")
SWARA_CHARS = "SsRrGgMmPpDdNn"
SIM_THRESHOLD = 0.85     # phrases with normalized similarity >= this merge


def phrase_labels(t):
    ph = t.phrases
    if ph is None:
        return []
    ev = getattr(ph, "events", None)
    return [str(e) for e in ev] if ev is not None else []


# ---- collect ----
recs = []
for tid in ds.track_ids:
    t = ds.track(tid)
    labs = phrase_labels(t)
    if not labs:
        continue
    raga = "?"
    md = t.metadata or {}
    raagas = md.get("raaga") or []
    if raagas and isinstance(raagas[0], dict):
        raga = raagas[0].get("name", "?")
    if raga in EXCLUDE:
        continue
    recs.append((tid, labs, raga))

ragas = [r for _, _, r in recs]
raga_counts = Counter(ragas)
evaluable = {r for r, c in raga_counts.items() if c >= 2}
print(f"Tracks: {len(recs)}   Evaluable ragas: {len(evaluable)}   "
      f"Queries: {sum(1 for r in ragas if r in evaluable)}")


# ============ approximate matching: cluster the phrase vocabulary ============
def norm_sim(a, b):
    """Normalized similarity via Levenshtein distance (1 - dist/maxlen).
    Simple DP edit distance; vocab is small enough (~535)."""
    if a == b:
        return 1.0
    la, lb = len(a), len(b)
    if la == 0 or lb == 0:
        return 0.0
    # early length-ratio gate (cheap): very different lengths can't be >=0.85
    if min(la, lb) / max(la, lb) < SIM_THRESHOLD:
        return 0.0
    prev = list(range(lb + 1))
    for i in range(1, la + 1):
        cur = [i] + [0] * lb
        for j in range(1, lb + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            cur[j] = min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + cost)
        prev = cur
    dist = prev[lb]
    return 1.0 - dist / max(la, lb)


def cluster_vocabulary(all_phrases):
    """Greedy single-linkage clustering: map each phrase to a canonical form.
    Sort by frequency so the most common variant becomes the cluster head."""
    freq = Counter(all_phrases)
    uniq = [p for p, _ in freq.most_common()]   # frequent first
    canon = {}                                   # phrase -> canonical head
    heads = []
    for p in uniq:
        placed = False
        # only compare against existing heads of similar length (speed)
        for h in heads:
            if abs(len(h) - len(p)) / max(len(h), len(p)) > (1 - SIM_THRESHOLD):
                continue
            if norm_sim(p, h) >= SIM_THRESHOLD:
                canon[p] = h
                placed = True
                break
        if not placed:
            heads.append(p)
            canon[p] = p
    return canon


all_phrases = [p for _, labs, _ in recs for p in labs]
print(f"Distinct phrases (exact): {len(set(all_phrases))}")
canon = cluster_vocabulary(all_phrases)
print(f"Canonical phrases (after approx merge @ {SIM_THRESHOLD}): "
      f"{len(set(canon.values()))}")


# ============ TF-IDF builder (shared) ============
def build_tfidf(docs):
    df = Counter()
    for d in docs:
        for p in set(d):
            df[p] += 1
    vocab = {p: i for i, p in enumerate(sorted(df))}
    N = len(docs)
    idf = np.zeros(len(vocab))
    for p, i in vocab.items():
        idf[i] = np.log((N + 1) / (df[p] + 1)) + 1
    X = np.zeros((N, len(vocab)))
    for r, d in enumerate(docs):
        for p, c in Counter(d).items():
            X[r, vocab[p]] = c * idf[vocab[p]]
    n = np.linalg.norm(X, axis=1, keepdims=True); n[n == 0] = 1
    return X / n


docs_exact = [labs for _, labs, _ in recs]
docs_approx = [[canon[p] for p in labs] for _, labs, _ in recs]

X_exact = build_tfidf(docs_exact)
X_approx = build_tfidf(docs_approx)

# bag of notes from chars
def bon(labs):
    v = Counter(ch for p in labs for ch in p if ch in SWARA_CHARS)
    vec = np.array([v[c] for c in SWARA_CHARS], dtype=float)
    s = vec.sum()
    return vec / s if s else vec
Xbon = np.array([bon(labs) for _, labs, _ in recs])
nb = np.linalg.norm(Xbon, axis=1, keepdims=True); nb[nb == 0] = 1
Xbon = Xbon / nb


# ============ eval ============
def evaluate(X, name, ks=(1, 3, 5)):
    sim = X @ X.T
    precisions = {k: [] for k in ks}
    aps, rrs = [], []
    ne = 0
    for i in range(len(ragas)):
        if ragas[i] not in evaluable:
            continue
        ne += 1
        order = np.argsort(-sim[i]); order = order[order != i]
        rel = np.array([ragas[j] == ragas[i] for j in order], dtype=int)
        for k in ks:
            precisions[k].append(rel[:k].sum() / k)
        if rel.sum():
            cum = np.cumsum(rel)
            pa = cum / (np.arange(len(rel)) + 1)
            aps.append((pa * rel).sum() / rel.sum())
            rrs.append(1.0 / (np.argmax(rel) + 1))
    row = {"representation": name, "evaluable_queries": ne}
    for k in ks:
        row[f"P@{k}"] = round(float(np.mean(precisions[k])), 3)
    row["MAP"] = round(float(np.mean(aps)), 3)
    row["MRR"] = round(float(np.mean(rrs)), 3)
    return row


results = [
    evaluate(Xbon, "bag_of_notes (chars)"),
    evaluate(X_exact, "phrase_vsm_exact"),
    evaluate(X_approx, f"phrase_vsm_approx (>={SIM_THRESHOLD})"),
]
df = pd.DataFrame(results)
print("\n=== PHRASE SUB-STUDY: exact vs approximate matching ===")
print(df.to_string(index=False))
print(f"\nvocab: {len(set(all_phrases))} exact -> {len(set(canon.values()))} clustered")
print("delta (approx - exact) P@1 quantifies transcription-variance cost.")
df.to_csv(os.path.join(OUT_DIR, "phrase_vsm_approx_results.csv"), index=False)
print(f"Saved -> {OUT_DIR}/phrase_vsm_approx_results.csv")

Tracks: 94   Evaluable ragas: 22   Queries: 60
Distinct phrases (exact): 535
Canonical phrases (after approx merge @ 0.85): 459

=== PHRASE SUB-STUDY: exact vs approximate matching ===
            representation  evaluable_queries   P@1   P@3   P@5   MAP   MRR
      bag_of_notes (chars)                 60 0.150 0.122 0.083 0.212 0.276
          phrase_vsm_exact                 60 0.333 0.139 0.100 0.291 0.391
phrase_vsm_approx (>=0.85)                 60 0.450 0.183 0.123 0.406 0.519

vocab: 535 exact -> 459 clustered
delta (approx - exact) P@1 quantifies transcription-variance cost.
Saved -> /content/census_out/phrase_vsm_approx_results.csv


In [9]:
"""
Phrase sub-study, Step 4 -- robustness: bootstrap CIs + threshold sweep
=============================================================
Converts the single-split phrase result into a defensible one:

  (A) BOOTSTRAP confidence intervals on P@1/MAP/MRR by resampling the 60
      queries with replacement -> is the phrase advantage real given small N?
  (B) THRESHOLD SWEEP for approximate matching (0.80..0.92) -> is the
      phrase_vsm_approx advantage stable, or an artifact of one threshold?

Reports bag_of_notes | phrase_vsm_exact | phrase_vsm_approx(each threshold)
with bootstrap mean +/- and 95% CI.
"""

import os
from collections import Counter

import numpy as np
import pandas as pd
import mirdata

DATA_HOME = "/content/saraga_carnatic"
OUT_DIR = "/content/census_out"
ds = mirdata.initialize("saraga_carnatic", data_home=DATA_HOME, version="1.5")

EXCLUDE = ("Rāgamālika", "Ragamalika", "?", "")
SWARA_CHARS = "SsRrGgMmPpDdNn"
THRESHOLDS = [0.80, 0.85, 0.90, 0.92]
N_BOOT = 1000
RNG = np.random.default_rng(42)


def phrase_labels(t):
    ph = t.phrases
    if ph is None:
        return []
    ev = getattr(ph, "events", None)
    return [str(e) for e in ev] if ev is not None else []


# ---- collect ----
recs = []
for tid in ds.track_ids:
    t = ds.track(tid)
    labs = phrase_labels(t)
    if not labs:
        continue
    raga = "?"
    md = t.metadata or {}
    rr = md.get("raaga") or []
    if rr and isinstance(rr[0], dict):
        raga = rr[0].get("name", "?")
    if raga in EXCLUDE:
        continue
    recs.append((tid, labs, raga))

ragas = np.array([r for _, _, r in recs])
raga_counts = Counter(ragas)
evaluable = {r for r, c in raga_counts.items() if c >= 2}
query_idx = [i for i in range(len(recs)) if ragas[i] in evaluable]
print(f"Tracks: {len(recs)}   Evaluable ragas: {len(evaluable)}   "
      f"Queries: {len(query_idx)}")


# ---- edit-distance similarity + clustering ----
def norm_sim(a, b):
    if a == b:
        return 1.0
    la, lb = len(a), len(b)
    if la == 0 or lb == 0:
        return 0.0
    if min(la, lb) / max(la, lb) < 0.75:
        return 0.0
    prev = list(range(lb + 1))
    for i in range(1, la + 1):
        cur = [i] + [0] * lb
        ai = a[i-1]
        for j in range(1, lb + 1):
            cost = 0 if ai == b[j-1] else 1
            cur[j] = min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + cost)
        prev = cur
    return 1.0 - prev[lb] / max(la, lb)


def cluster_vocab(all_phrases, thr):
    freq = Counter(all_phrases)
    heads = []
    canon = {}
    for p in (x for x, _ in freq.most_common()):
        placed = False
        for h in heads:
            if abs(len(h) - len(p)) / max(len(h), len(p)) > (1 - thr):
                continue
            if norm_sim(p, h) >= thr:
                canon[p] = h; placed = True; break
        if not placed:
            heads.append(p); canon[p] = p
    return canon


# ---- TF-IDF ----
def build_tfidf(docs):
    df = Counter()
    for d in docs:
        for p in set(d):
            df[p] += 1
    vocab = {p: i for i, p in enumerate(sorted(df))}
    N = len(docs)
    idf = np.array([np.log((N + 1) / (df[p] + 1)) + 1 for p in vocab])
    X = np.zeros((N, len(vocab)))
    for r, d in enumerate(docs):
        for p, c in Counter(d).items():
            X[r, vocab[p]] = c * idf[vocab[p]]
    n = np.linalg.norm(X, axis=1, keepdims=True); n[n == 0] = 1
    return X / n


# ---- per-query metric arrays (so we can bootstrap over queries) ----
def per_query_metrics(X):
    sim = X @ X.T
    p1, ap, rr = [], [], []
    for i in query_idx:
        order = np.argsort(-sim[i]); order = order[order != i]
        rel = (ragas[order] == ragas[i]).astype(int)
        p1.append(rel[:1].sum())
        if rel.sum():
            cum = np.cumsum(rel)
            pa = cum / (np.arange(len(rel)) + 1)
            ap.append((pa * rel).sum() / rel.sum())
            rr.append(1.0 / (np.argmax(rel) + 1))
        else:
            ap.append(0.0); rr.append(0.0)
    return np.array(p1, float), np.array(ap, float), np.array(rr, float)


def boot_ci(arr):
    idx = RNG.integers(0, len(arr), size=(N_BOOT, len(arr)))
    means = arr[idx].mean(axis=1)
    return arr.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)


docs_exact = [labs for _, labs, _ in recs]
all_phrases = [p for d in docs_exact for p in d]

def bon(labs):
    v = Counter(ch for p in labs for ch in p if ch in SWARA_CHARS)
    vec = np.array([v[c] for c in SWARA_CHARS], float)
    s = vec.sum(); return vec / s if s else vec
Xbon = np.array([bon(l) for _, l, _ in recs])
nb = np.linalg.norm(Xbon, axis=1, keepdims=True); nb[nb == 0] = 1
Xbon = Xbon / nb

reps = {"bag_of_notes": Xbon,
        "phrase_vsm_exact": build_tfidf(docs_exact)}
vocab_sizes = {"phrase_vsm_exact": len(set(all_phrases))}
for thr in THRESHOLDS:
    canon = cluster_vocab(all_phrases, thr)
    docs_a = [[canon[p] for p in labs] for _, labs, _ in recs]
    reps[f"phrase_approx@{thr}"] = build_tfidf(docs_a)
    vocab_sizes[f"phrase_approx@{thr}"] = len(set(canon.values()))

rows = []
for name, X in reps.items():
    p1, ap, rr = per_query_metrics(X)
    m1, lo1, hi1 = boot_ci(p1)
    ma, loa, hia = boot_ci(ap)
    mr, lor, hir = boot_ci(rr)
    rows.append({
        "representation": name,
        "vocab": vocab_sizes.get(name, "-"),
        "P@1": f"{m1:.3f} [{lo1:.3f},{hi1:.3f}]",
        "MAP": f"{ma:.3f} [{loa:.3f},{hia:.3f}]",
        "MRR": f"{mr:.3f} [{lor:.3f},{hir:.3f}]",
    })

df = pd.DataFrame(rows)
print(f"\n=== PHRASE SUB-STUDY robustness ({N_BOOT} bootstrap resamples, "
      f"95% CI over {len(query_idx)} queries) ===")
print(df.to_string(index=False))
print("\nRead:")
print(" - if phrase_vsm_exact P@1 CI lower bound > bag_of_notes P@1 upper bound,")
print("   the phrase advantage is significant despite small N.")
print(" - if phrase_approx P@1 stays high & similar across thresholds, the")
print("   approximate-matching gain is robust, not a threshold artifact.")
df.to_csv(os.path.join(OUT_DIR, "phrase_vsm_robust.csv"), index=False)
print(f"Saved -> {OUT_DIR}/phrase_vsm_robust.csv")

Tracks: 94   Evaluable ragas: 22   Queries: 60

=== PHRASE SUB-STUDY robustness (1000 bootstrap resamples, 95% CI over 60 queries) ===
    representation vocab                 P@1                 MAP                 MRR
      bag_of_notes     - 0.150 [0.067,0.250] 0.212 [0.148,0.286] 0.276 [0.202,0.366]
  phrase_vsm_exact   535 0.333 [0.217,0.450] 0.291 [0.198,0.384] 0.391 [0.286,0.504]
 phrase_approx@0.8   418 0.450 [0.317,0.567] 0.406 [0.315,0.506] 0.538 [0.426,0.646]
phrase_approx@0.85   459 0.450 [0.333,0.583] 0.406 [0.300,0.517] 0.519 [0.409,0.633]
 phrase_approx@0.9   526 0.350 [0.233,0.467] 0.300 [0.203,0.404] 0.399 [0.288,0.521]
phrase_approx@0.92   531 0.350 [0.233,0.467] 0.300 [0.205,0.406] 0.399 [0.290,0.510]

Read:
 - if phrase_vsm_exact P@1 CI lower bound > bag_of_notes P@1 upper bound,
   the phrase advantage is significant despite small N.
 - if phrase_approx P@1 stays high & similar across thresholds, the
   approximate-matching gain is robust, not a threshold artifact.